[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/eigen_fluids/eigen_fluids.ipynb)

# Laplacian Eigenfunction Fluids

Laplacian eigenfunction fluids describe a surface flow as a combination of spatial modes. The surface's geometry determines both the modes and how they interact; the evolving state is a set of amplitudes.

The flow follows a streamfunction's level curves, with velocity `gradient(stream) ^ normals`. Pairing gradients with these tangent flows gives the interactions that transfer motion between modes.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import islice

import numpy as np

from numga import stack
from numga.algebras import VGA3D as ga
from numga.sparse import spdiag
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.fluids.eigen_fluids import render
from examples.fluids.eigen_fluids.scenarios import bunny

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar, context = mesh_module.Mesh, mesh_module.as_scalar, mesh_module.ctx
mv = context.multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Streamfunction gradients on triangles are extrinsic planes across the gradient direction.
Plane = ga.gatype.vector()                          # Vector+
# Vertex positions, edge displacements, and tangent fluid velocities are intrinsic antivectors.
Point = ga.gatype.antivector()                      # Antivector-
# Oriented triangle area elements and unit tangent patches are joins of two edges.
Patch = ga.gatype.antibivector()                    # Antibivector-
GradientMap = ga.gatype((Plane, Scalar))
Superposition = ga.gatype((Scalar, Scalar))
Bracket = ga.gatype((Scalar, Scalar, Scalar))

## Flow modes on the bunny

A streamfunction acts like a height field whose contours guide the flow: steeper slopes mean faster motion along a contour. Its smoothest spatial patterns are the Laplace–Beltrami modes. The bunny's shape determines where their broad currents and finer eddies can fit.

In [ ]:
mesh, vertices = bunny()
save_figure(render.draw_surface(mesh, vertices), "eigen_fluids_surface");

In [ ]:
# Count of non-constant Laplace–Beltrami modes, and the four modes shown in the gallery.
n_modes = 128
shown_modes = np.array([0, 7, 31, 119])

In [ ]:
# The oriented edge opposite each corner of each triangle, the triangle's area patch, its area, and its unit tangent patch.
sides = mesh.boundary[2, 1] * (mesh.d[0] * vertices)                            # [corners] Point-[faces]
patches = (sides[0] & sides[1]) / 2                                             # Patch-[faces]
areas = patches.norm()                                                          # Scalar+[faces]
normals = patches / areas                                                       # Patch-[faces]

# The corner hat gradients assemble into the Whitney gradient map from vertex scalars to face gradient planes.
corner_gradients = (sides | patches.inverse()) / 2                              # [corners] Plane+[faces]
gradient = (spdiag(corner_gradients * Scalar) * mesh.incidence[2, 0]).sum(axis=0)  # (Plane+ <- Scalar+)[faces <- vertices]
# Each face's mean over its corners; its reverse hands each corner a third of every face around it.
corner_mean = mesh.incidence[2, 0].mean(axis=0)                                 # Scalar+[faces <- vertices]
dual_areas = ~corner_mean * areas                                               # Scalar+[vertices]

# Pulling the area-weighted metric dual of face gradients back to the vertices gives the Dirichlet stiffness form.
stiffness = gradient.adjugate()(spdiag(areas * Plane.dual())(gradient))         # (Amount- <- Scalar+)[vertices <- vertices]
mass = spdiag(dual_areas * Scalar.dual())                                       # (Amount- <- Scalar+)[vertices <- vertices]

The modes are the eigenvectors of the stiffness against the mass, the smoothest first; the constant field costs no energy and carries no flow.

In [ ]:
# Solve for the lowest Laplace–Beltrami modes and drop the constant null mode. The eigenvalues, like the amplitudes
# of the modes later, are fields over the modes, which the mode interactions couple.
eigenvalues, eigenfields = stiffness.eigh(mass, n_modes + 1)                    # [modes + 1] Scalar+, [modes + 1] Scalar+[vertices]
eigenvalues, modes = eigenvalues[1:].field(), eigenfields[1:]                   # Scalar+[modes], [modes] Scalar+[vertices]
# The surface field of modal amplitudes superposes the modes they weigh.
superposition = (modes * Scalar).batch().field(0, 1).adjoint()                 # (Scalar+ <- Scalar+)[vertices <- modes]

# Meeting each mode's gradient plane with the unit tangent patch gives its divergence-free velocity on every face.
mode_velocities = gradient(modes[shown_modes]) ^ normals                        # [shown] Point-[faces]
save_figure(render.draw_modes(mesh, vertices, modes[shown_modes], mode_velocities), "eigen_fluids_modes");

Colour shows each mode's streamfunction, and the contours trace its flow paths. The modes progress from broad patterns across the body to smaller structures shaped by the ears and limbs.

In finite element notation, the modes satisfy $K\phi_i=\lambda_i M\phi_i$, where $K$ is the Dirichlet stiffness and $M$ the vertex mass matrix.

In [ ]:
print("selected mode eigenvalues:", np.round(eigenvalues.batch()[shown_modes].to_array(), 3))

## Mode interactions

The flow's nonlinearity is the advection of vorticity by the flow, the Lie–Poisson bracket of streamfunction and vorticity. A field's gradient reads its change along a flow through the regressive pairing with velocity, so in the modes the bracket becomes one map of two arguments, `structure(stream, vorticity)`, which depends only on the geometry and is computed once.

In [ ]:
def structure_constants(mesh: Mesh, patches: Patch, gradient: GradientMap, modes: Scalar) -> Bracket:
    """Each modal Lie–Poisson structure constant is the surface integral of a mode against the Jacobian of two
    others, the wedge of their gradients read on the surface. Bound to a streamfunction and a vorticity, the
    constants give the advection rate of every mode."""
    # Each face's mean over its corners.
    corner_mean = mesh.incidence[2, 0].mean(axis=0)                             # Scalar+[faces <- vertices]
    face_gradients = gradient(modes)                                            # [modes] Plane+[faces]
    # Meeting each mode's gradient with the triangle's area patch gives its area-weighted tangent velocity,
    # which pairs against another gradient to give the Jacobian integrated over the triangle.
    flows = face_gradients ^ patches                                            # [modes] Point-[faces]
    jacobians = face_gradients[:, None] & flows[None, :]                        # [modes, modes] Scalar+[faces]
    # Against a Jacobian constant on each triangle, a mode integrates through its mean over the triangle's corners.
    face_means = corner_mean * modes                                            # [modes] Scalar+[faces]
    integral = (face_means * Scalar).batch().field(0, 1)                        # (Scalar+ <- Scalar+)[modes <- faces]
    # Totally antisymmetric, the constants are unchanged by a cyclic shift of their three modes: the mode the
    # Jacobians are integrated against may stand last, as the mode of the vorticity.
    return (integral(jacobians).batch() * Scalar * Scalar).field(0, 1, 2)      # (Scalar+ <- (Scalar+, Scalar+))[modes <- (modes, modes)]


structure = structure_constants(mesh, patches, gradient, modes)               # (Scalar+ <- (Scalar+, Scalar+))[modes <- (modes, modes)]

## Interacting vortices

Nearby patches of opposite circulation form vortex dipoles; their interaction drives motion across the curved surface.

In [ ]:
# Six opposing vortex dipoles distributed across the bunny's shoulder, chest, flank, haunch, neck, and back.
vortex_centres = mv.antivector([
    [0.35, -0.54, -0.03],
    [0.40, -0.69, -0.29],
    [-0.08, -0.52, 0.03],
    [-0.16, -0.58, -0.27],
    [-0.54, -0.48, 0.02],
    [-0.45, -0.58, -0.39],
    [0.34, -0.31, 0.26],
    [0.09, -0.40, 0.20],
    [0.22, 0.34, -0.16],
    [-0.14, 0.28, -0.08],
    [-0.48, 0.26, -0.14],
    [0.06, 0.47, -0.30],
])                                                  # [vortices] Point-
vortex_strengths = np.array([75.0, -75.0, -75.0, 75.0, 70.0, -70.0, -65.0, 65.0, 72.0, -72.0, 68.0, -68.0])
vortex_radius = 0.12
eigenvalue_cutoff = 140.0

# Fourth-order Runge–Kutta step size, total step count, frame stride, kinematic viscosity, and frame duration.
dt = 0.007
n_steps = 800
stride = 8
viscosity = 0.0009
frame_duration_ms = 40

In [ ]:
def vorticity_rate(vorticity: Scalar, structure: Bracket, eigenvalues: Scalar, viscosity: float) -> Scalar:
    """The modal vorticity amplitudes change at the rate of the Lie–Poisson advection bracket of streamfunction
    and vorticity, less viscous diffusion."""
    stream = vorticity / eigenvalues                                            # Scalar+[modes]
    advection = structure(stream, vorticity)                                    # Scalar+[modes]
    return advection - viscosity * eigenvalues * vorticity                      # Scalar+[modes]


def advect(initial: Scalar, structure: Bracket, eigenvalues: Scalar, viscosity: float,
           dt: float, n_steps: int) -> Iterator[Scalar]:
    """Successive modal vorticities follow classical fourth-order Runge–Kutta steps."""
    vorticity = initial                                                         # Scalar+[modes]
    yield vorticity
    for _ in range(n_steps):
        first = vorticity_rate(vorticity, structure, eigenvalues, viscosity)                       # Scalar+[modes]
        second = vorticity_rate(vorticity + (dt / 2) * first, structure, eigenvalues, viscosity)   # Scalar+[modes]
        third = vorticity_rate(vorticity + (dt / 2) * second, structure, eigenvalues, viscosity)   # Scalar+[modes]
        fourth = vorticity_rate(vorticity + dt * third, structure, eigenvalues, viscosity)         # Scalar+[modes]
        vorticity = vorticity + (dt / 6) * (first + 2 * second + 2 * third + fourth)               # Scalar+[modes]
        yield vorticity


# Project the initial localized vortex dipoles onto the Laplace–Beltrami modes with a smooth spectral taper.
squared_distances = (vertices.batch()[:, None] - vortex_centres[None, :]).norm_squared()   # [vertices, vortices] Scalar+
spots = ((squared_distances * (-0.5 / vortex_radius**2)).exp() * vortex_strengths).sum(axis=-1).field()   # Scalar+[vertices]
taper = (-0.5 * (eigenvalues / eigenvalue_cutoff) ** 2).exp()                   # Scalar+[modes]
initial_vorticity = (modes & mass(spots)).sites.sum().field() * taper           # Scalar+[modes]

inviscid = stack(tuple(advect(initial_vorticity, structure, eigenvalues, 0.0, dt, n_steps)))   # [steps + 1] Scalar+[modes]

In [ ]:
def reconstruct(states: Iterable[Scalar], superposition: Superposition, eigenvalues: Scalar,
                gradient: GradientMap, normals: Patch) -> Iterator[tuple[Scalar, Scalar, Point]]:
    """Each modal state gives the surface vorticity, streamfunction, and tangent velocity of one frame."""
    for amplitudes in states:
        vorticity = superposition(amplitudes)                                   # Scalar+[vertices]
        stream = superposition(amplitudes / eigenvalues)                        # Scalar+[vertices]
        velocity = gradient(stream) ^ normals                                   # Point-[faces]
        yield vorticity, stream, velocity

frames = reconstruct(inviscid[::stride], superposition, eigenvalues, gradient, normals)
show_animation(render.animate_flow(mesh, vertices, frames), "eigen_fluids_bunny", frame_duration_ms)

Warm and cool patches mark opposite circulation. The streamfunction contours show the flow paths that carry them. The evolving amplitudes describe the whole surface, so a local vortex can interact with motion far from its starting point.

## Energy transfer and viscous decay

The mode interactions redistribute energy and enstrophy, the integrated squared vorticity. Their total antisymmetry makes the inviscid rate pair to zero with both streamfunction and vorticity. Viscosity damps modes in proportion to their eigenvalues, removing fine structure more quickly than broad currents.

In [ ]:
# The inviscid run of the animation, against the same start under viscosity.
viscous = stack(tuple(advect(initial_vorticity, structure, eigenvalues, viscosity, dt, n_steps)))   # [steps + 1] Scalar+[modes]
runs = stack((inviscid, viscous))                                        # [cases, steps + 1] Scalar+[modes]
energies = (runs.squared() / eigenvalues).sites.sum() / 2                # [cases, steps + 1] Scalar+
enstrophies = runs.squared().sites.sum() / 2                             # [cases, steps + 1] Scalar+
inviscid_energy, viscous_energy = energies                               # [steps + 1] Scalar+, Scalar+
inviscid_enstrophy, viscous_enstrophy = enstrophies                      # [steps + 1] Scalar+, Scalar+

times = np.arange(n_steps + 1) * dt
save_figure(render.draw_conservation(times, inviscid_energy, viscous_energy, inviscid_enstrophy,
                                     viscous_enstrophy, initial_vorticity, inviscid[-1], viscous[-1]),
            "eigen_fluids_conservation");

The dark curves show inviscid transport; green shows viscous energy and orange viscous enstrophy. The right panel shows how enstrophy is distributed across the modes. Enstrophy gives finer modes more weight than energy does, so the viscous flow loses a larger fraction of its enstrophy.

In the Lie–Poisson notation of model-reduced fluid dynamics, $C_{kij}=\int_M\phi_k\,d\phi_i\wedge d\phi_j$ and $\dot\omega_k=\sum_{i,j}C_{kij}(\omega_i/\lambda_i)\omega_j-\nu\lambda_k\omega_k$. The modal formulation follows De Witt, Lessig and Fiume, [*Fluid Simulation using Laplacian Eigenfunctions*](https://www.dgp.toronto.edu/~tyler/eigenfluids/) (2012), and Liu, Mason, Tong and Desbrun, [*Model-Reduced Variational Fluid Simulation*](https://geometry.caltech.edu/pubs/LMTD15.pdf) (2015).

## Shear layers on a sphere

Opposite bands of vorticity make the flow easier to read than on a complicated surface. A meridional ripple disturbs the bands; the evolving contours show how the flow carries that disturbance. The geometric mesh is finer than the mode set, and the same reduced RK4 step uses only the precomputed interaction constants and eigenvalues.

In [ ]:
# Complete spherical harmonic bands through degree eight, on a finer geometric mesh.
n_sphere_levels = 4
n_sphere_modes = 80
sphere_latitude = 0.44
sphere_width = 0.16
sphere_strength = 9.0
sphere_wave_amplitude = 0.065
sphere_dt = 0.025
n_sphere_steps = 400
sphere_stride = 4
sphere_viscosity = 0.001

In [ ]:
sphere_mesh, sphere_vertices = Mesh.icosphere(n_sphere_levels)
# Each opposite edge and its face patch determine the corner hat gradient.
sphere_sides = sphere_mesh.boundary[2, 1] * (sphere_mesh.d[0] * sphere_vertices)   # [corners] Point-[faces]
sphere_patches = (sphere_sides[0] & sphere_sides[1]) / 2                    # Patch-[faces]
sphere_areas = sphere_patches.norm()                                        # Scalar+[faces]
sphere_normals = sphere_patches / sphere_areas                              # Patch-[faces]
sphere_corner_gradients = (sphere_sides | sphere_patches.inverse()) / 2     # [corners] Plane+[faces]
sphere_gradient = (spdiag(sphere_corner_gradients * Scalar) * sphere_mesh.incidence[2, 0]).sum(axis=0)   # (Plane+ <- Scalar+)[faces <- vertices]
# Each face's mean over its corners; its reverse hands each corner a third of every face around it.
sphere_dual_areas = ~sphere_mesh.incidence[2, 0].mean(axis=0) * sphere_areas   # Scalar+[vertices]

# The stiffness and vertex mass determine the surface's spatial modes.
sphere_stiffness = sphere_gradient.adjugate()(spdiag(sphere_areas * Plane.dual())(sphere_gradient))   # (Amount- <- Scalar+)[vertices <- vertices]
sphere_mass = spdiag(sphere_dual_areas * Scalar.dual())                     # (Amount- <- Scalar+)[vertices <- vertices]
sphere_eigenvalues, sphere_eigenfields = sphere_stiffness.eigh(sphere_mass, n_sphere_modes + 1)   # [modes + 1] Scalar+, [modes + 1] Scalar+[vertices]
sphere_eigenvalues, sphere_modes = sphere_eigenvalues[1:].field(), sphere_eigenfields[1:]   # Scalar+[modes], [modes] Scalar+[vertices]
sphere_superposition = (sphere_modes * Scalar).batch().field(0, 1).adjoint()   # (Scalar+ <- Scalar+)[vertices <- modes]
sphere_structure = structure_constants(sphere_mesh, sphere_patches, sphere_gradient, sphere_modes)   # (Scalar+ <- (Scalar+, Scalar+))[modes <- (modes, modes)]

In [ ]:
def shear_layers(vertices: Point, dual_areas: Scalar, latitude: float, width: float,
                 strength: float, wave_amplitude: float) -> Scalar:
    """Opposite vorticity bands with a meridional ripple and zero total circulation."""
    points = vertices.batch()                                                  # [vertices] Point-
    heights = mv.z & points                                                    # [vertices] Scalar+
    n_meridians = 3
    angles = np.pi * np.arange(n_meridians) / n_meridians
    meridians = (mv.xy * (-angles / 2)).exp() >> mv.x                          # [meridians] Plane+
    first, second, third = meridians[:, None] & points                         # [vertices] Scalar+, Scalar+, Scalar+
    # Three meridian readings make the azimuthal ripple vanish smoothly at the poles.
    shift = -4 * wave_amplitude * first * second * third                       # [vertices] Scalar+
    north = (heights - np.sin(latitude) - shift) / width                       # [vertices] Scalar+
    south = (heights + np.sin(latitude) - shift) / width                       # [vertices] Scalar+
    vorticity = (strength * ((-0.5 * north.squared()).exp() - (-0.5 * south.squared()).exp())).field()   # Scalar+[vertices]
    return vorticity - (dual_areas * vorticity).sites.sum() / dual_areas.sites.sum()


shear_field = shear_layers(sphere_vertices, sphere_dual_areas, sphere_latitude, sphere_width,
                           sphere_strength, sphere_wave_amplitude)          # Scalar+[vertices]
sphere_start = (sphere_modes & sphere_mass(shear_field)).sites.sum().field()   # Scalar+[modes]
sphere_states = islice(advect(sphere_start, sphere_structure, sphere_eigenvalues, sphere_viscosity, sphere_dt,
                              n_sphere_steps), 0, None, sphere_stride)
sphere_frames = reconstruct(sphere_states, sphere_superposition, sphere_eigenvalues, sphere_gradient, sphere_normals)
show_animation(render.animate_flow(sphere_mesh, sphere_vertices, sphere_frames), "eigen_fluids_sphere", frame_duration_ms)

Warm and cool bands mark opposite circulation. The mesh and eigensolve are setup costs; during the animation each stage sums the modal interaction constants against the amplitudes, and surface fields are reconstructed only for displayed frames.

In [ ]:
# checks
# The Laplace–Beltrami eigenmodes are orthonormal with respect to the lumped vertex mass form.
gram = (modes[:, None] & mass(modes[None, :])).sites.sum().to_array()
np.testing.assert_allclose(gram, np.eye(n_modes), atol=1e-12)
# Every mode velocity and reconstructed flow velocity lies in its triangle's tangent plane.
np.testing.assert_allclose((mode_velocities & patches).to_array(), 0.0, atol=1e-14)
_, _, final_velocity = next(reconstruct(inviscid[-1:], superposition, eigenvalues, gradient, normals))
np.testing.assert_allclose((final_velocity & patches).to_array(), 0.0, atol=1e-13)
# On every triangle, the Jacobian wedge pairing equals the area-weighted directional derivative.
face_gradients = gradient(modes[:6])                                            # [subset] Plane+[faces]
wedge_brackets = (face_gradients[:, None] ^ face_gradients[None, :]) & patches  # [subset, subset] Scalar+[faces]
directional = (face_gradients[:, None] & (face_gradients[None, :] ^ normals)) * areas   # [subset, subset] Scalar+[faces]
np.testing.assert_allclose((wedge_brackets - directional).to_array(), 0.0, atol=1e-14)
# The Lie–Poisson structure constants are totally antisymmetric in their three modes to machine precision.
constants = structure.kernel[..., 0, 0, 0]                                      # [modes, modes, modes]
np.testing.assert_allclose(constants + constants.swapaxes(1, 2), 0.0, atol=1e-13)
np.testing.assert_allclose(constants + constants.swapaxes(0, 1), 0.0, atol=1e-13)
# Instantaneous inviscid advection produces zero kinetic energy and zero enstrophy, and RK4 keeps both flat.
initial_rate = vorticity_rate(initial_vorticity, structure, eigenvalues, 0.0)   # Scalar+[modes]
np.testing.assert_allclose(((initial_vorticity / eigenvalues) * initial_rate).sites.sum().to_array(), 0.0, atol=1e-11)
np.testing.assert_allclose((initial_vorticity * initial_rate).sites.sum().to_array(), 0.0, atol=1e-11)
energy_trace = inviscid_energy.to_array()
enstrophy_trace = inviscid_enstrophy.to_array()
np.testing.assert_allclose(energy_trace, energy_trace[0], rtol=5e-5)
np.testing.assert_allclose(enstrophy_trace, enstrophy_trace[0], rtol=5e-5)
# Viscosity dissipates both invariants, damping high-wavenumber enstrophy faster than kinetic energy.
viscous_energy_trace = viscous_energy.to_array()
viscous_enstrophy_trace = viscous_enstrophy.to_array()
assert np.all(np.diff(viscous_energy_trace) < 0)
assert np.all(np.diff(viscous_enstrophy_trace) < 0)
assert viscous_enstrophy_trace[-1] / viscous_enstrophy_trace[0] < viscous_energy_trace[-1] / viscous_energy_trace[0]